# Gulf Stream Drifter Data: Processing and EDA

**Goal:** Prepare continuous trajectories for Member 3 and count available forecast windows.

**Pipeline:** GDP hourly data → Regional filtering → Quality checks → Segmentation → EDA and window counts → Export.

**Study region: 25–45°N, 80–50°W** (a rectangular study window, not the Gulf Stream axis).

**Run:** Execute from top to bottom. The first scan needs internet access; cached blocks support resuming. Regional data are loaded into memory. Use `MAX_OBSERVATIONS=None` for full regional statistics.


In [ ]:
# Uncomment and run the next line if dependencies are missing; then restart the kernel.
# %pip install clouddrift numpy pandas matplotlib pyarrow xarray


## 1. Set parameters
Set the region, a 24-hour history and 1/24/168-hour forecasts. Outputs go to `member2_outputs/`, separated by region and scan configuration.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REGION = dict(lon_min=-80., lon_max=-50., lat_min=25., lat_max=45.)
CHUNK_SIZE = 250_000
MAX_WORKERS = 8              # Parallel extraction threads; reduce to 2 if memory is limited.
MAX_OBSERVATIONS = None      # Use 1_000_000 for debugging; use None for full EDA.
HISTORY_HOURS = 24
HORIZONS = (1, 24, 168)
SAMPLE_DRIFTERS = 100
RANDOM_SEED = 42
EXPORT_FULL = True

HERE = Path.cwd()
if (HERE / 'member2' / 'gulf_stream_eda.ipynb').exists():
    HERE = HERE / 'member2'
assert (HERE / 'gulf_stream_eda.ipynb').exists(), 'Start the notebook from the repository root or member2 directory.'
assert -180 <= REGION['lon_min'] < REGION['lon_max'] <= 180
assert -90 <= REGION['lat_min'] < REGION['lat_max'] <= 90
assert CHUNK_SIZE > 0 and HISTORY_HOURS >= 0
assert isinstance(MAX_WORKERS, int) and MAX_WORKERS >= 1
assert MAX_OBSERVATIONS is None or MAX_OBSERVATIONS > 0
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})


## 2. Read GDP hourly data
Check fields, units and versions. Map observations to drifter IDs using `rowsize`, without loading the entire global dataset.


In [ ]:
from clouddrift.datasets import gdp1h

ds = gdp1h()   # On-demand data access through CloudDrift
required = ['id', 'rowsize', 'time', 'lat', 'lon']
missing_required = [v for v in required if v not in ds]
if missing_required:
    raise ValueError(f'Missing required fields: {missing_required}')
rowsize = np.asarray(ds['rowsize'].values, dtype=np.int64)
ids = np.asarray(ds['id'].values)
ends = np.cumsum(rowsize)
assert len(rowsize) == len(ids) and (rowsize >= 0).all()
assert len(ends) and ends[-1] == ds.sizes['obs']
assert not pd.isna(ids).any() and pd.Index(ids).is_unique, 'Drifter IDs must be non-null and unique.'
for name in ['time', 'lat', 'lon']:
    assert ds[name].dims == ('obs',), (name, ds[name].dims)

OPTIONAL = ['ve', 'vn', 'sst', 'gap', 'drogue_status',
            'err_lat', 'err_lon', 'err_ve', 'err_vn']
VARIABLES = ['time', 'lat', 'lon'] + OPTIONAL
schema = pd.DataFrame([
    dict(variable=v, available=v in ds,
         dimensions=str(ds[v].dims) if v in ds else '',
         units=ds[v].attrs.get('units', '') if v in ds else '',
         description=ds[v].attrs.get('long_name', '') if v in ds else '')
    for v in ['id', 'rowsize'] + VARIABLES
])
display(schema)
print('Dataset dimensions:', dict(ds.sizes))
print('Dataset attributes:', ds.attrs)


In [ ]:
scan_stop = min(ds.sizes['obs'], MAX_OBSERVATIONS or ds.sizes['obs'])
complete_scan = scan_stop == ds.sizes['obs']
config = dict(schema_version=1, region=REGION, chunk_size=CHUNK_SIZE,
              scan_stop=int(scan_stop), total_obs=int(ds.sizes['obs']),
              source_attributes=dict(ds.attrs), clouddrift_version=version('clouddrift'),
              variables=VARIABLES)
signature = hashlib.sha256(json.dumps(config, sort_keys=True, default=str).encode()).hexdigest()[:12]
RUN_DIR = HERE / 'member2_outputs' / (('full_' if complete_scan else 'partial_') + signature)
CACHE = RUN_DIR / 'cache'
FIGURES = RUN_DIR / 'figures'
for directory in [CACHE, FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)
schema.to_csv(RUN_DIR / 'variable_metadata.csv', index=False)
STATUS = 'Full global scan' if complete_scan else 'PARTIAL SCAN — not regional totals'
print(STATUS, '| Output:', RUN_DIR)


## 3. Extract the study region
Filter and cache coordinate blocks, preserving source indices to detect trajectory breaks. Missing coordinates are excluded, so regional missingness does not represent global missingness.

Parallel extraction uses `MAX_WORKERS` threads and reuses cached blocks. Progress shows elapsed time and an estimated time remaining, which varies with reading speed.


In [ ]:
def extract_block(dataset, start, stop, trajectory_ends, buoy_ids, region):
    lat = np.asarray(dataset['lat'].isel(obs=slice(start, stop)).values)
    lon = (np.asarray(dataset['lon'].isel(obs=slice(start, stop)).values) + 180) % 360 - 180
    inside = (np.isfinite(lat) & np.isfinite(lon)
              & (lat >= region['lat_min']) & (lat <= region['lat_max'])
              & (lon >= region['lon_min']) & (lon <= region['lon_max']))
    local = np.flatnonzero(inside)
    source = start + local
    trajectory = np.searchsorted(trajectory_ends, source, side='right')
    data = dict(source_obs=source, trajectory_index=trajectory,
                buoy_id=buoy_ids[trajectory], lat=lat[local], lon=lon[local])
    for name in ['time'] + OPTIONAL:
        if name not in dataset:
            data[name] = np.full(len(local), np.nan)
        elif not len(local):
            data[name] = np.array([], dtype='datetime64[ns]' if name == 'time' else float)
        elif dataset[name].dims == ('obs',):
            # Read only this block, without loading the global array.
            data[name] = np.asarray(dataset[name].isel(obs=slice(start, stop)).values)[local]
        elif dataset[name].dims == ('traj',):
            data[name] = np.asarray(dataset[name].values)[trajectory]
        else:
            raise ValueError(f'Check the dimensions of {name}: {dataset[name].dims}')
    return pd.DataFrame(data)

from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from time import perf_counter
from IPython.display import HTML

def format_duration(seconds):
    if seconds is None:
        return 'Estimating'
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def show_progress(handle, stage, done, total, elapsed, remaining, detail=''):
    percent = 100 * done / total if total else 100
    handle.update(HTML(
        f'<b>{stage}</b>: {done:,}/{total:,} blocks ({percent:.1f}%)<br>'
        f'<progress value="{done}" max="{max(total, 1)}" style="width:100%"></progress><br>'
        f'Elapsed {format_duration(elapsed)} | ETA {format_duration(remaining)}'
        f'<br>{detail}'
    ))

def cache_block(start, path):
    block = extract_block(ds, start, min(start + CHUNK_SIZE, scan_stop), ends, ids, REGION)
    temporary = path.with_suffix('.tmp')
    block.to_parquet(temporary, index=False)
    temporary.replace(path)  # Reuse a cache block only after writing it completely.


def extract_pending(jobs, workers, on_progress):
    # Bound in-flight tasks; update progress on the main thread, and read/write caches in workers.
    jobs = iter(jobs)
    pool = ThreadPoolExecutor(max_workers=workers)
    pending = {}
    completed = 0
    started = perf_counter()

    def submit_next():
        job = next(jobs, None)
        if job is not None:
            start, path = job
            pending[pool.submit(cache_block, start, path)] = start

    try:
        for _ in range(workers):
            submit_next()
        while pending:
            finished, _ = wait(pending, timeout=1, return_when=FIRST_COMPLETED)
            for future in finished:
                start = pending.pop(future)
                try:
                    future.result()
                except Exception as exc:
                    raise RuntimeError(f'Extraction failed for observation block {start:,}; completed caches can be reused after fixing the issue.') from exc
                completed += 1
            for _ in finished:
                submit_next()
            on_progress(completed, perf_counter() - started, len(pending))
    except BaseException:
        for future in pending:
            future.cancel()
        print('Stopping: waiting for active blocks to finish; completed caches will be retained.', flush=True)
        raise
    finally:
        # Wait for active writers to finish so reruns cannot write to the same cache concurrently.
        pool.shutdown(wait=True, cancel_futures=True)


step_started = perf_counter()
starts = range(0, scan_stop, CHUNK_SIZE)
paths = [CACHE / f'{start:012d}.parquet' for start in starts]
jobs = [(start, path) for start, path in zip(starts, paths) if not path.exists()]
new_total = len(jobs)
cached_count = len(paths) - new_total
progress = display(HTML('Preparing parallel extraction…'), display_id=True)

def update_extraction(completed, elapsed, active):
    remaining = elapsed / completed * (new_total - completed) if completed else None
    if completed == new_total:
        remaining = 0
    show_progress(progress, 'Regional extraction', cached_count + completed, len(paths),
                  perf_counter() - step_started, remaining,
                  f'Cached {cached_count:,} blocks; completed this run {completed:,}/{new_total:,}; '
                  f'In flight {active} blocks (up to {MAX_WORKERS} threads)')

update_extraction(0, 0, 0)
extract_pending(jobs, MAX_WORKERS, update_extraction)

read_started = perf_counter()
read_progress = display(HTML('Reading regional caches…'), display_id=True)
def read_cached_blocks():
    for part, path in enumerate(paths, 1):
        frame = pd.read_parquet(path)
        elapsed = perf_counter() - read_started
        show_progress(read_progress, 'Reading caches', part, len(paths), elapsed,
                      elapsed / part * (len(paths) - part))
        yield frame

regional = pd.concat(read_cached_blocks(), ignore_index=True)
print('Cache reading complete; sorting and checking regional data…', flush=True)
regional = regional.sort_values('source_obs').reset_index(drop=True)
if regional.empty:
    raise ValueError('No regional observations in this scan; a debug prefix does not establish that the whole region is empty.')
regional['time'] = pd.to_datetime(regional['time'], errors='coerce', utc=True).dt.tz_localize(None)
assert regional['source_obs'].is_unique
print(f'{STATUS}: {len(regional):,} observations, {regional.buoy_id.nunique():,} drifters')
print(f'Regional table memory: {regional.memory_usage(deep=True).sum()/1024**2:,.1f} MiB')
print(f'Total step time: {format_duration(perf_counter() - step_started)}')
display(regional.head())


## 4. Check data quality
Count missing values, infinities and duplicate timestamps. Do not impute optional fields or remove records by a speed threshold. Exclude invalid timestamps and all duplicate `(buoy_id, time)` rows before segmentation.


In [ ]:
def missing_table(frame, names, available):
    records = []
    for name in names:
        values = frame[name]
        mask = values.isna()
        if pd.api.types.is_numeric_dtype(values):
            mask = mask | ~np.isfinite(values)
        records.append(dict(variable=name, status='available' if name in available else 'not in source',
                            missing_count=int(mask.sum()), missing_pct=100 * mask.mean()))
    return pd.DataFrame(records)

missing_values = missing_table(regional, VARIABLES, set(ds.variables))
duplicate_time = regional.duplicated(['buoy_id', 'time'], keep=False)
invalid_time = regional.time.isna()
same_buoy = regional.trajectory_index.eq(regional.trajectory_index.shift())
quality_audit = pd.DataFrame([
    ['regional_observations', len(regional)],
    ['missing_time_rows', int(invalid_time.sum())],
    ['duplicate_buoy_time_rows', int((duplicate_time & ~invalid_time).sum())],
    ['non_increasing_time_steps_in_source_order',
     int((same_buoy & regional.time.diff().le(pd.Timedelta(0))).sum())],
], columns=['metric', 'value'])
display(missing_values)
display(quality_audit)


## 5. Build continuous segments
Continuity requires **the same drifter + adjacent source indices + exactly one-hour intervals**, assessed in source order without rounding timestamps.

Gaps and regional exits break segments. A 168-hour span needs 169 points; both history and future must remain inside the region.


In [ ]:
def build_segments(frame):
    ordered = frame.sort_values('source_obs').copy()
    duplicated = ordered.duplicated(['buoy_id', 'time'], keep=False)
    clean = ordered.loc[ordered.time.notna() & ~duplicated].copy()
    if clean.empty:
        raise ValueError('No regional observations with valid, unique timestamps.')
    connected = (clean.trajectory_index.eq(clean.trajectory_index.shift())
                 & clean.source_obs.diff().eq(1)
                 & clean.time.diff().eq(pd.Timedelta(hours=1)))
    clean['segment_id'] = (~connected).cumsum().astype('int64')
    segments = clean.groupby('segment_id', sort=True).agg(
        buoy_id=('buoy_id', 'first'), n_obs=('source_obs', 'size'),
        start_time=('time', 'min'), end_time=('time', 'max'))
    segments['duration_hours'] = (segments.end_time - segments.start_time).dt.total_seconds()/3600
    assert np.array_equal(segments.duration_hours.to_numpy(), segments.n_obs.to_numpy() - 1)
    clean = clean.sort_values(['buoy_id', 'time', 'source_obs']).reset_index(drop=True)
    return clean, segments.reset_index()

clean, segments = build_segments(regional)
per_drifter = regional.groupby('buoy_id').agg(
    regional_observations=('source_obs', 'size'),
    first_time=('time', 'min'), last_time=('time', 'max'))
per_drifter = per_drifter.join(segments.groupby('buoy_id').agg(
    segment_count=('segment_id', 'size'),
    max_continuous_hours=('duration_hours', 'max'),
    median_segment_hours=('duration_hours', 'median'),
    mean_segment_hours=('duration_hours', 'mean'),
    q25_segment_hours=('duration_hours', lambda x: x.quantile(.25)),
    q75_segment_hours=('duration_hours', lambda x: x.quantile(.75))))
per_drifter['usable_observations'] = clean.groupby('buoy_id').size().reindex(per_drifter.index, fill_value=0)
display(per_drifter.head())
display(per_drifter[['regional_observations', 'max_continuous_hours']].describe())
display(segments[['n_obs', 'duration_hours']].describe())


## 6. Count coverage and forecast windows
Count annual observations, segments spanning 24/48/168 hours and eligible forecast origins.

For `N` points, `L` history hours and `H` forecast hours: **eligible origins = max(N − L − H, 0)**. Overlapping windows are not independent samples.


In [ ]:
yearly_counts = (regional.dropna(subset=['time']).assign(year=lambda x: x.time.dt.year)
                 .groupby('year').size().rename('observations').to_frame())
if not yearly_counts.empty:
    yearly_counts = yearly_counts.reindex(range(int(yearly_counts.index.min()),
                                               int(yearly_counts.index.max()) + 1), fill_value=0)
yearly_counts.index.name = 'year'

continuity_thresholds = pd.DataFrame([
    dict(required_duration_hours=h, minimum_observations=h+1,
         eligible_segments=int(segments.duration_hours.ge(h).sum()),
         eligible_drifters=int(segments.loc[segments.duration_hours.ge(h), 'buoy_id'].nunique()))
    for h in (24, 48, 168)
])

def forecast_counts(segment_table, history_hours, horizons):
    rows = []
    for horizon in horizons:
        for history in sorted(set([0, history_hours])):
            windows = (segment_table.n_obs - history - horizon).clip(lower=0)
            rows.append(dict(horizon_hours=horizon, history_hours=history,
                             minimum_observations=history+horizon+1,
                             eligible_segments=int(windows.gt(0).sum()),
                             eligible_drifters=int(segment_table.loc[windows.gt(0), 'buoy_id'].nunique()),
                             candidate_origins=int(windows.sum())))
    return pd.DataFrame(rows)

forecast_feasibility = forecast_counts(segments, HISTORY_HOURS, HORIZONS)
summary = pd.DataFrame([
    ['scan_status', STATUS], ['unique_regional_drifters', regional.buoy_id.nunique()],
    ['regional_observations', len(regional)], ['usable_observations', len(clean)],
    ['first_observation_utc', str(regional.time.min())],
    ['last_observation_utc', str(regional.time.max())],
    ['median_observations_per_drifter', per_drifter.regional_observations.median()],
    ['continuous_segments', len(segments)],
    ['median_continuous_segment_hours', segments.duration_hours.median()],
    ['drifters_with_24_continuous_hours', int(per_drifter.max_continuous_hours.ge(24).sum())],
    ['drifters_with_168_continuous_hours', int(per_drifter.max_continuous_hours.ge(168).sum())],
], columns=['metric', 'value'])
display(summary)
display(continuity_thresholds)
display(forecast_feasibility)


## 7. Inspect velocity and SST
Summarise velocity and quality fields, retaining extremes. Base quality thresholds on source documentation; check whether velocity estimates use future information before modelling.


In [ ]:
def units(name):
    return str(ds[name].attrs.get('units', '')) if name in ds else ''

velocity_units = units('ve')
if 've' in ds and 'vn' in ds and units('ve') != units('vn'):
    raise ValueError('ve/vn units differ; define a conversion rule first.')
finite_velocity = np.isfinite(regional.ve) & np.isfinite(regional.vn)
regional['speed'] = np.where(finite_velocity, np.hypot(regional.ve, regional.vn), np.nan)
speed_statistics = regional[['ve', 'vn', 'speed']].replace([np.inf, -np.inf], np.nan).describe(
    percentiles=[.25, .5, .75, .95, .99]).T
speed_statistics['units'] = [units('ve'), units('vn'), velocity_units or 'unknown']
display(speed_statistics)
for name in ['gap', 'drogue_status']:
    print(f'\n{name}: source metadata =', dict(ds[name].attrs) if name in ds else 'not in source')
    display(regional[name].value_counts(dropna=False).head(12).rename('observations').to_frame())


## 8. Plot EDA figures
Show trajectories, spatial density, segment durations, annual coverage and speed/SST distributions. Plot the 100 drifters with the most observations, connecting points within each segment; use all regional observations for density.


In [ ]:
def save_figure(fig, name):
    fig.savefig(FIGURES / name, dpi=180, bbox_inches='tight')
    plt.show()

selected_ids = per_drifter.nlargest(100, 'regional_observations').index
plot_data = clean[clean.buoy_id.isin(selected_ids)]
fig, ax = plt.subplots(figsize=(10, 6))
colors = {buoy: plt.cm.turbo(i / max(1, len(selected_ids)-1)) for i, buoy in enumerate(selected_ids)}
for (_, buoy), track in plot_data.groupby(['segment_id', 'buoy_id']):
    ax.plot(track.lon, track.lat, color=colors[buoy], linewidth=.5, alpha=.65)
ax.set(xlim=(REGION['lon_min'], REGION['lon_max']), ylim=(REGION['lat_min'], REGION['lat_max']),
       xlabel='Longitude (degrees east)', ylabel='Latitude (degrees north)',
       title=f'Gulf Stream region: top {len(selected_ids)} drifters\n{STATUS}')
ax.set_aspect(1 / np.cos(np.deg2rad((REGION['lat_min'] + REGION['lat_max'])/2)))
ax.grid(alpha=.2)
save_figure(fig, '01_drifter_trajectories.png')

fig, ax = plt.subplots(figsize=(9, 5))
hb = ax.hexbin(regional.lon, regional.lat, gridsize=65, bins='log', mincnt=1, cmap='viridis')
fig.colorbar(hb, ax=ax, label='Observation count (log colour scale)')
ax.set(xlabel='Longitude (degrees east)', ylabel='Latitude (degrees north)',
       title=f'All regional observations: spatial coverage\n{STATUS}')
save_figure(fig, '02_spatial_coverage.png')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
# Use log1p to retain zero-hour segments and show the long tail; tick labels remain in hours.
ax.hist(np.log1p(segments.duration_hours), bins=50, color='#278c9e', edgecolor='white')
for h, color in [(24, '#df8f1d'), (168, '#b83c50')]:
    ax.axvline(np.log1p(h), color=color, linestyle='--', label=f'{h} hours')
ticks = [h for h in [0, 1, 6, 24, 48, 168, 720, 2160, 8760] if h <= max(168, segments.duration_hours.max())]
ax.set_xticks(np.log1p(ticks), labels=ticks)
ax.set(xlabel='Continuous elapsed hours (log1p scale)', ylabel='Number of segments',
       title=f'Continuous trajectory duration\n{STATUS}')
ax.legend()
save_figure(fig, '03_continuous_duration.png')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(yearly_counts.index, yearly_counts.observations, color='#278c9e')
ax.set(xlabel='Year (UTC)', ylabel='Regional observations', title=f'Temporal coverage\n{STATUS}')
save_figure(fig, '04_observations_by_year.png')

fig, ax = plt.subplots(figsize=(9, 4))
speeds = regional.speed.dropna()
if len(speeds):
    ax.hist(speeds, bins=80, color='#5278a5')
    ax.set_yscale('log')
else:
    ax.text(.5, .5, 'No finite velocity pairs available', transform=ax.transAxes, ha='center')
ax.set(xlabel=f'Speed ({velocity_units or "source units unavailable"})', ylabel='Observations (log scale)',
       title=f'Surface current speed — full range retained\n{STATUS}')
save_figure(fig, '05_speed_distribution.png')

sst = pd.to_numeric(regional.sst, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
if len(sst):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(sst, bins=60, color='#cd8550')
    ax.set(xlabel=f'SST ({units("sst") or "unknown units"})', ylabel='Observations',
           title=f'Sea surface temperature (optional)\n{STATUS}')
    save_figure(fig, '06_sst_distribution.png')


## 9. Export modelling data
Export summary tables, full data and a development subset sampled by drifter. Times are UTC; rows are sorted by `buoy_id, time`.

- Keep each window within one `segment_id`, which is unique only within this output. Recheck continuity after feature filtering.
- Split training/validation/test sets by **buoy_id**. Fit imputation, scaling and mean flow using training data only.
- The development sample is not a dataset split. Forecasts outside the region require additional trajectory extraction.


In [ ]:
for table, filename in [
    (summary, 'gulf_stream_summary.csv'),
    (forecast_feasibility, 'forecast_feasibility.csv'),
    (continuity_thresholds, 'continuity_thresholds.csv'),
    (per_drifter.reset_index(), 'per_drifter_summary.csv'),
    (segments, 'continuous_segments.csv'),
    (missing_values, 'missing_values.csv'),
    (quality_audit, 'quality_audit.csv'),
    (yearly_counts.reset_index(), 'yearly_observation_counts.csv'),
    (speed_statistics.rename_axis('variable').reset_index(), 'speed_statistics.csv'),
]:
    table.to_csv(RUN_DIR / filename, index=False)

model_columns = ['buoy_id', 'time', 'lat', 'lon', 've', 'vn', 'sst', 'gap',
                 'drogue_status', 'err_lat', 'err_lon', 'err_ve', 'err_vn',
                 'segment_id', 'source_obs', 'trajectory_index']
# Use nullable boolean dtype to avoid object-to-bool changes when reading Parquet back.
status_values = clean['drogue_status'].dropna().unique()
if not all(isinstance(value, (bool, np.bool_)) for value in status_values):
    raise ValueError('drogue_status contains non-boolean values; check the source encoding first.')
clean['drogue_status'] = clean['drogue_status'].astype('boolean')

rng = np.random.default_rng(RANDOM_SEED)
available_ids = clean.buoy_id.unique()
sample_ids = rng.choice(available_ids, size=min(SAMPLE_DRIFTERS, len(available_ids)), replace=False)
model_sample = clean.loc[clean.buoy_id.isin(sample_ids), model_columns].copy()
model_sample.to_parquet(RUN_DIR / 'gulf_stream_model_sample.parquet', index=False)
if EXPORT_FULL:
    clean[model_columns].to_parquet(RUN_DIR / 'gulf_stream_full.parquet', index=False)

manifest = dict(config, generated_utc=datetime.now(timezone.utc).isoformat(),
                complete_scan=complete_scan, status=STATUS,
                regional_observations=len(regional), usable_observations=len(clean),
                region_confirmed_by_member1=False, history_hours=HISTORY_HOURS,
                forecast_horizons_hours=list(HORIZONS), sample_seed=RANDOM_SEED,
                sample_drifters=len(sample_ids), sample_observations=len(model_sample),
                time_convention='UTC; timezone-naive datetime64 in Parquet',
                continuity='Same drifter; adjacent source index; exactly one-hour interval',
                spatial_scope='All history and future points must stay within region',
                package_versions={name: version(name) for name in
                                  ['clouddrift', 'numpy', 'pandas', 'matplotlib', 'pyarrow', 'xarray']})
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
# Read back to verify that export preserves row count, order and fields.
roundtrip = pd.read_parquet(RUN_DIR / 'gulf_stream_model_sample.parquet')
pd.testing.assert_frame_equal(roundtrip, model_sample.reset_index(drop=True))
assert not roundtrip.duplicated(['buoy_id', 'time']).any()
print('Saved outputs:', RUN_DIR)
print('Sample:', len(model_sample), 'rows;', len(sample_ids), 'drifters')


## 10. Generate the data summary
Summarise the region, record counts, segments and one-week forecast windows. Only a complete scan supports regional totals; window counts do not measure prediction accuracy.


In [ ]:
week = forecast_feasibility.query('horizon_hours == 168 and history_hours == @HISTORY_HOURS').iloc[0]
proposal_text = (
    f'{STATUS}. Within the Gulf Stream study box '
    f'{REGION["lat_min"]}–{REGION["lat_max"]}°N and '
    f'{abs(REGION["lon_min"])}–{abs(REGION["lon_max"])}°W, '
    f'the GDP hourly data contain {len(regional):,} regional observations from '
    f'{regional.buoy_id.nunique():,} drifters, spanning '
    f'{regional.time.min()} to {regional.time.max()} UTC. '
    f'After excluding invalid and duplicated timestamps, '
    f'{len(segments):,} strictly hourly segments remain, with a median elapsed duration '
    f'of {segments.duration_hours.median():.1f} hours. '
    f'For a {HISTORY_HOURS}-hour history and a 168-hour forecast, '
    f'{int(week.eligible_drifters):,} drifters provide '
    f'{int(week.candidate_origins):,} overlapping candidate origins. '
    'Windows remain inside the box and overlap; they do not measure prediction accuracy. '
)
print(proposal_text)
(RUN_DIR / 'proposal_data_summary.txt').write_text(proposal_text + '\n', encoding='utf-8')


## 11. Handoff and references
Deliver Parquet files, `manifest.json`, field metadata and the feasibility table. Member 1 provides the regional justification; Member 3 confirms features, windows and quality rules (`gap`, uncertainty and drogue status). `gap` is not the timestamp interval.

References: [CloudDrift](https://clouddrift.org/) · [NOAA GDP](https://www.aoml.noaa.gov/phod/gdp/) · [GDP hourly v2.01](https://doi.org/10.25921/x46c-3620). Use source metadata for the actual version and units.
